# ЛР 02 — Синтетическая классификация: когда какая модель выигрывает?

## Цель
**Сгенерировать** синтетические датасеты, на которых каждая из моделей из набора становится лучшей
(по выбранной вами метрике):

- **LDA** — `LinearDiscriminantAnalysis`  
- **QDA** — `QuadraticDiscriminantAnalysis`  
- **Gaussian Naive Bayes** — `GaussianNB`  
- **Logistic Regression** — `LogisticRegression`

Требуется написать генераторы `make_dataset_*.

## Что сдаём
1. Ноутбук, в котором есть:
   - 4 генератора данных (по одному на “победу” каждой модели);
   - единый протокол оценки (код дан ниже) и итоговые таблицы;
   - визуализации границ решений для 2D-датасетов;
   - краткие объяснения (**почему** выиграла именно эта модель).
2. Короткий текст (1–2 страницы прямо в ноутбуке достаточно):
   - какие предположения модели выполняются/нарушаются;
   - какую метрику вы оптимизируете (**accuracy** или **log-loss**) и почему;
   - проверка устойчивости (несколько `random_state`).

## Метрики
- **Accuracy** (0–1 потери)
- **Log-loss** (качество вероятностей)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

## Модели (фиксированный набор)
Разрешается тюнить **по одному** гиперпараметру на модель (маленькая сетка),
но семейство моделей фиксировано.

In [ ]:
models = {
    "LDA": LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.0),
    "GaussianNB": GaussianNB(),
    "LogReg": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500))
    ])
}

## Единый протокол сравнения (можно использовать как есть)
- Кросс-валидация `StratifiedKFold`
- Метрики: accuracy и log-loss
- (Опционально) повторить для нескольких seed и усреднить

In [ ]:
def evaluate_models(X, y, models, cv_splits=5, random_state=0):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    scoring = {"acc": "accuracy", "nll": "neg_log_loss"}  # nll = negative log-loss

    rows = []
    for name, model in models.items():
        out = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
        rows.append({
            "model": name,
            "acc_mean": float(np.mean(out["test_acc"])),
            "acc_std": float(np.std(out["test_acc"])),
            "logloss_mean": float(-np.mean(out["test_nll"])),   # обратно в +logloss
            "logloss_std": float(np.std(-out["test_nll"]))
        })
    # Сортировка: лучший logloss (меньше) -> выше; затем accuracy (больше)
    rows = sorted(rows, key=lambda r: (r["logloss_mean"], -r["acc_mean"]))
    return rows

def print_results(rows, title=None):
    if title:
        print("\n" + title)
        print("-"*len(title))
    for r in rows:
        print(f"{r['model']:10s} | acc {r['acc_mean']:.3f} ± {r['acc_std']:.3f} | "
              f"logloss {r['logloss_mean']:.3f} ± {r['logloss_std']:.3f}")

## Визуализация границ решений (для 2D датасетов)
Используйте только если `X.shape[1]==2`.

In [ ]:
def plot_boundaries_2d(X, y, models, h=0.05, figsize=(22, 5), title=None):
    assert X.shape[1] == 2, "plot_boundaries_2d работает только для 2D датасетов."

    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    plt.figure(figsize=figsize)
    if title:
        plt.suptitle(title)

    for i, (name, clf) in enumerate(models.items()):
        ax = plt.subplot(1, len(models), i + 1)
        clf.fit(X, y)
        Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.25)
        ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", s=20)
        ax.set_title(name)

    plt.tight_layout()
    plt.show()

# Задание: написать 4 генератора данных

Реализуйте 4 функции ниже. Каждая должна возвращать `X, y`.

**Требование:** на каждом датасете лучшей (по выбранной метрике) должна быть *разная* модель.

Обоснуйте дизайн через предположения моделей:

- **LDA:** гауссовские классы + **общая** ковариация
- **QDA:** гауссовские классы + **разные** ковариации по классам
- **GaussianNB:** гауссовские классы + **условная независимость признаков** (диагональная ковариация)
- **LogReg:** верная **условная** модель `P(y=1|x)=σ(w^T x + b)` даже если `p(x|y)` не гауссовское

> Вы свободны выбирать: `n`, `d`, приоры классов, уровень шума, heavy tails, смеси и т.д.

In [ ]:
def make_dataset_for_lda(random_state=0):
    """Идея: гауссовские классы с ОДНОЙ полной ковариацией (с корреляциями)."""
    rng = np.random.default_rng(random_state)
    # TODO: реализовать (return X, y)
    raise NotImplementedError

def make_dataset_for_qda(random_state=0):
    """Идея: гауссовские классы с РАЗНЫМИ ковариациями."""
    rng = np.random.default_rng(random_state)
    # TODO: реализовать (return X, y)
    raise NotImplementedError

def make_dataset_for_gnb(random_state=0):
    """Идея: высокая размерность, условно независимые гауссовские признаки (диагональная ковариация)."""
    rng = np.random.default_rng(random_state)
    # TODO: реализовать (return X, y)
    raise NotImplementedError

def make_dataset_for_logreg(random_state=0):
    """Идея: x ~ негауссовское распределение, затем y|x ~ Bernoulli(sigmoid(w^T x + b))."""
    rng = np.random.default_rng(random_state)
    # TODO: реализовать (return X, y)
    raise NotImplementedError

## Подсказки (словами): как “подстроить” датасет под победителя

### Чтобы выиграл LDA
- Гауссовские “облака”, **общая** ковариация `Σ`, причём `Σ` должна иметь ненулевые вне диагонали (корреляции).
- Средние разнесены умеренно.
- Почему: LDA совпадает с истинной моделью; GaussianNB ошибается из-за независимости.

### Чтобы выиграл QDA
- Гауссовские классы, но формы облаков разные: `Σ0 != Σ1` (разные вытянутости/повороты).
- Средние можно сделать близкими, чтобы линейная граница была плоха.
- Почему: истинная граница квадратична; LDA слишком жёсткая.

### Чтобы выиграл GaussianNB
- `d` большой, `n` небольшой.
- Признаки условно независимы: диагональные ковариации.
- Много “слабых” информативных координат, которые NB суммирует.
- Почему: LDA/QDA оценивать ковариации сложно/неустойчиво, NB устойчив.

### Чтобы выиграл LogReg
- Генерация по условной модели `P(y=1|x)=σ(w^T x+b)`.
- Сделайте `x` негауссовским (смесь/тяжёлые хвосты), чтобы вредить генеративным предположениям.
- Почему: дискриминативная модель верна по `p(y|x)`, а `p(x|y)` у генеративных мисспецифицирована.

## Запуск и сравнение (заполнить после реализации генераторов)

Для каждого датасета:
1) сгенерировать `X, y`
2) (если 2D) нарисовать границы решений
3) посчитать таблицу `evaluate_models`
4) написать объяснение: какие предположения выполняются/нарушаются

In [ ]:
def run_one_dataset(make_fn, name, seeds=(0, 1, 2), do_plot_if_2d=True):
    all_rows = []
    for s in seeds:
        X, y = make_fn(random_state=s)

        if do_plot_if_2d and X.shape[1] == 2 and s == seeds[0]:
            plot_boundaries_2d(X, y, models, title=f"{name} (seed={s})")

        rows = evaluate_models(X, y, models, cv_splits=5, random_state=s)
        all_rows.append(rows)

    # усреднение по seed: берём среднее от CV-средних
    agg = {}
    for rows in all_rows:
        for r in rows:
            agg.setdefault(r["model"], {"acc": [], "logloss": []})
            agg[r["model"]]["acc"].append(r["acc_mean"])
            agg[r["model"]]["logloss"].append(r["logloss_mean"])

    summary = []
    for m, v in agg.items():
        summary.append({
            "model": m,
            "acc_mean": float(np.mean(v["acc"])),
            "acc_std": float(np.std(v["acc"])),
            "logloss_mean": float(np.mean(v["logloss"])),
            "logloss_std": float(np.std(v["logloss"])),
        })
    summary = sorted(summary, key=lambda r: (r["logloss_mean"], -r["acc_mean"]))
    print_results(summary, title=f"Итог по seed для: {name}")
    return summary

# Пример (раскомментировать после реализации генераторов):
# run_one_dataset(make_dataset_for_lda, "Датасет для LDA")

## Комментарии к результатам (итоговые выводы)

Ниже — “шпаргалка”, как интерпретировать результаты и что именно демонстрирует каждый синтетический датасет.

### Почему для Logistic Regression используется scaling, а для LDA/QDA/NB — обычно нет
- **LogisticRegression** оптимизирует log-loss с регуляризацией (например, L2). Без стандартизации признаки разного масштаба
  ухудшают численную устойчивость оптимизации и делают регуляризацию “несправедливой” (штраф по весам зависит от масштаба фич).
  Поэтому `StandardScaler` — стандартная практика.
- **LDA/QDA/GaussianNB** — генеративные гауссовские модели: масштаб признаков “учитывается” в оценённых дисперсиях/ковариациях.
  Стандартизация не обязательна для корректности (хотя иногда помогает численно).

### Что именно ломалось на NB-датасете и почему
- На варианте с **d ≫ n** падал **QDA**: в каждом классе эмпирическая ковариация имеет ранг ≤ n_k−1, то есть не может быть
  полной (и, следовательно, неинвертируема).
- Это *не баг*, а математическое ограничение: QDA пытается оценить полную матрицу ковариаций размера d×d для каждого класса,
  что требует существенно больше данных.
- Два корректных сценария для лабы:
  1) **“Режим применимости”**: показать, что без регуляризации/структуры QDA в high-d режиме может быть неприменим.
     Тогда протокол сравнения должен уметь отмечать `fail` (например, NaN).
  2) **“Сравнение без падений”**: выбрать умеренно высокую размерность (например d≈50–80 при n≈200–400), чтобы все модели обучались,
     и при этом GaussianNB выигрывал за счёт меньшего числа параметров.

### Интерпретация датасетов
#### QDA-датасет (класс-зависимые ковариации)
- Истина: классы гауссовские, но **Σ0 ≠ Σ1**.
- Граница решений в общем случае **квадратична**.
- Поэтому **QDA** выигрывает (и по accuracy, и по log-loss), а LDA/LogReg (линейные границы) уступают.

#### GNB-датасет (условная независимость + high-d)
- Истина: признаки (почти) **условно независимы** при фиксированном классе (диагональные ковариации).
- В высоких размерностях GaussianNB устойчив, потому что оценивает всего ~K·d дисперсий/средних.
- QDA оценивает ~K·d(d+1)/2 параметров ковариации ⇒ вероятности могут стать плохо калиброванными
  (типичный симптом — большой **log-loss**, даже если accuracy не катастрофический).

#### LogReg-датасет (истинная условная модель)
- Данные порождены из **условной** модели `P(y=1|x)=σ(w^T x + b)`.
- Логистическая регрессия корректно специфицирована по `p(y|x)` ⇒ обычно лучшая по **log-loss**.
- LDA может оказаться близко, если маргинальное распределение x “похоже” на гауссовское и граница почти линейна.

#### LDA-датасет (общая ковариация)
- Если истинная модель — LDA (общая Σ), то LDA “правильная”.
- QDA более гибкая и иногда может быть почти столь же хороша (или даже немного лучше по accuracy) из‑за конечной выборки,
  но по **log-loss** LDA обычно не хуже и часто чуть лучше/стабильнее.
- LogReg тоже линейный классификатор, поэтому при почти линейной границе может быть очень близко к LDA.

### Как “усилить” демонстрацию (идеи для творческой части)
- Чтобы **LogReg** сильнее обогнала LDA по log-loss: сделать x более **негауссовским** (смесь, heavy tails, “кольца”),
  при этом сохранить истинный логит `σ(w^T x + b)`.
- Чтобы **GaussianNB** ещё увереннее выигрывал: увеличить число независимых слабых информативных признаков
  (много маленьких сдвигов средних), оставить n умеренным.
- Чтобы **QDA** выигрывал ярче: сделать различие ковариаций более выраженным (повороты/вытянутости), а средние — ближе.

---
**Главная идея лабы:** “какая модель лучшая” определяется тем, насколько её предположения совпадают с механизмом генерации данных,
и насколько устойчиво оцениваются параметры в выбранном режиме (n, d, шум, дисбаланс).